![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/customers_orders.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
orders_df = (
    spark.readStream
            .table("orders_silver")
)
display(orders_df, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/orders_silver_{time.time()}")

In [0]:
%sql
select *
from table_changes("customers_silver",2)

In [0]:
cdf_customers_df = (
    spark.readStream
            .format("delta")
            .option("readChangeData", True)
            .option("startingVersion",3)
            .table("customers_silver")
            .filter(F.col("_change_type").isin(["insert","update_postimage"]))
)
display(cdf_customers_df, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/cdf_customers_silver_{time.time()}")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customers_orders
(order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country STRING, row_time TIMESTAMP, processed_timestamp TIMESTAMP)

In [0]:
query = (
    orders_df.join(cdf_customers_df, orders_df.customer_id == cdf_customers_df.customer_id,"inner")
    .drop(cdf_customers_df["_change_type"],cdf_customers_df["_commit_version"],cdf_customers_df["customer_id"])        
)
display(query, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/orders_silver_joined_{time.time()}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def batch_upsert(microBatchDF, batchId):
    window = Window.partitionBy("order_id","customer_id").orderBy(F.col("processed_timestamp").desc())
    (
        microBatchDF
        .withColumn("rank", F.rank().over(window))
        .filter("rank = 1")
        .drop("rank")
        .createOrReplaceTempView("ranked_updates")
    )

    query = """
        merge into customers_orders a 
        using ranked_updates b 
        on a.customer_id=b.customer_id and a.order_id=b.order_id
        when matched and a.processed_timestamp < b.processed_timestamp
            then update set *
        when not matched
            then insert *

    """
    microBatchDF.sparkSession.sql(query)

In [0]:
def process_customers_orders():
    orders_df = (
            spark.readStream
                    .table("orders_silver")
                    )
    
    cdf_customers_df = (
            spark.readStream
                    .format("delta")
                    .option("readChangeData", True)
                    .option("startingVersion",3)
                    .table("customers_silver")
                    .filter(F.col("_change_type").isin(["insert","update_postimage"]))
            )
    
    query = (orders_df.join(cdf_customers_df, orders_df.customer_id == cdf_customers_df.customer_id,"inner")
                        .drop(cdf_customers_df["_change_type"],cdf_customers_df["_commit_version"],cdf_customers_df["customer_id"])
                        .withColumnRenamed("_commit_timestamp","processed_timestamp")
                        .writeStream
                        .foreachBatch(batch_upsert)
                        .option("checkpointLocation", f"{bookstore.checkpoint_path}/customers_orders")
                        .trigger(availableNow=True)
                        .start()
             )
    query.awaitTermination()
    #display(query, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/customers_orders_{time.time()}")

In [0]:
process_customers_orders()

In [0]:
%sql
select *
from customers_orders